In [ ]:
# Enable auto-reloading of custom modules
%load_ext autoreload
%autoreload 2

# Core Python and data tools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd

# Mesa model components
import mesa
from mesa import model
from model.traffic_model import TrafficModel

# Custom utility modules (aliased for clarity)
import utils.unit_conversion_utils as uc
import utils.analysis_utils as au
import utils.animation_utils as anim

# Visualization and display (Jupyter-specific)
from IPython.display import display, HTML

print(mesa.__version__)

# Load external data

## Road

In [ ]:
#road_gdf = gpd.read_file("data/roads/hw210_w_speed_limits.geojson")
road_gdf = gpd.read_parquet("data/roads/hw210_sl_and_curvs.parquet")

#au.plot_colored_road(road_gdf.loc[road_gdf.curvature>15], 'curvature')
au.plot_colored_road(road_gdf, 'speed_limit')

## Expected Counts Seconds

In [ ]:
ecs_df = pd.read_csv('data/vehicle_counts/expected_counts_seconds.csv')

# sns.lineplot(ecs_df[['100','90','80', '60','40' ]])
# plt.title('p_generate over time')
# # plt.show()

# Single Model Run

In [ ]:
%%time
model = TrafficModel(
    # meta perams
    road_gdf=road_gdf, ecs_df=ecs_df, max_steps=10800, batchrun=False, collect_every_n=5,
    # car centric perams
     start_hr=7, traffic_percentile=80, p_generate=.7, max_persons=10000,
    #bus centric perams
    bus_interval=15, car_preference=.7
)

model.run_model()

print(f'Model ran for {model.steps} steps')

In [ ]:
model.too_close_counter

# Analysis

In [ ]:
# process the finished_agents data 
finished_agents = au.finished_agents_summary_df(model, plots=True)
vehicles_full = au.vehicle_agent_data_time_series(model, plots=True)
model_ts = au.model_data_time_series(model)

au.plot_speed_delta(vehicles_full,model_ts)

In [ ]:
# currently usefull for volume_by_section, speed_by_section, density_by_section
au.plot_section_trends(model_ts, col='density_by_section', window=1)

# currently usefull for speed_change, speed_change_mps2, speed, speed_mps, gap_m, ideal_gap_m
#au.plot_mean_feature(vehicles_full, 'gap_m')


# Animations

In [ ]:
# run the animation
# looking at one car
issue_car_id =  None
issue_step = 0

In [ ]:
anim.animate_traffic(vehicles_full, road_gdf, interval=100, step_skip=10, watch=issue_car_id, zoom=20)

In [ ]:
anim.animate_relative_distance(vehicle_df=vehicles_full, agent_id=issue_car_id, distance_behind=100)

# Issue Car Analysis

In [ ]:
au.plot_single_car_driving_actions(vehicles_full, issue_car_id)

In [ ]:
step_range=(500,600)
au.plot_agent_trajectories(vehicles_full, [issue_car_id, issue_car_id+1], 'speed',step_range)